# Smile Detection System Using Haar Cascade

Detect human faces and smiles from images and webcam input using OpenCV Haar Cascade classifiers.

## 1. Project Objective

* Detect human faces in an image or webcam frame.
* Detect smiles **inside** the detected faces.
* Draw bounding boxes around faces and smiles.
* Display the label **"Smile"** when a smile is found.
* Demonstrate both image-based and webcam-based detection.
* Deploy the system through a Streamlit web app (`app.py`).

No model is trained in this project. We only use OpenCV's pretrained Haar Cascade files.

## 2. Introduction

Smile detection is a common computer vision task used in cameras and photo apps. Here we solve it with a **classical** technique, a two-step pipeline:

```
Image -> Grayscale -> Detect faces -> For each face -> Detect smile -> Draw boxes
```

### Why Haar Cascade?

Haar Cascade is a classical computer vision technique that uses pretrained classifiers to detect specific objects or features. In this project, pretrained classifiers are used to detect faces and smiles without training a new machine learning model.

## 3. Technologies Used

* **Python**
* **OpenCV** - image processing and Haar Cascade classifiers
* **NumPy** - image arrays
* **Matplotlib** - showing images inside the notebook
* **Streamlit** - web interface (in `app.py`)

## 4. Import Libraries

In [13]:
import os

import cv2
import numpy as np
import matplotlib.pyplot as plt

## 5. Load Haar Cascade Models

`cv2.data.haarcascades` is the folder where OpenCV stores its built-in XML files, so we do not need to download anything. 

The XML files are required because a Haar Cascade classifier is stored as an XML file containing the pretrained feature values. Without it, the classifier is empty and cannot detect anything. We check `.empty()` to make sure loading worked.

In [15]:

import cv2
import streamlit as st


@st.cache_resource
def load_cascades():
    """Load OpenCV's pretrained Haar Cascade classifiers."""

    face_path = (
        cv2.data.haarcascades
        + "haarcascade_frontalface_default.xml"
    )

    smile_path = (
        cv2.data.haarcascades
        + "haarcascade_smile.xml"
    )

    face = cv2.CascadeClassifier(face_path)
    smile = cv2.CascadeClassifier(smile_path)

    if face.empty():
        raise RuntimeError(
            f"Face Cascade could not be loaded: {face_path}"
        )

    if smile.empty():
        raise RuntimeError(
            f"Smile Cascade could not be loaded: {smile_path}"
        )

    return face, smile


try:
    face_cascade, smile_cascade = load_cascades()
    st.success("Face and smile Haar Cascades loaded successfully.")

except Exception as e:
    st.error(f"Failed to load Haar Cascades: {e}")
    st.stop()


2026-10-10 10:15:01.886 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 10:15:01.886 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 10:15:01.890 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## 6. Face Detection and Smile Detection (reusable function)

`detect_faces_and_smiles()` does the whole job:

1. Converts the BGR frame to grayscale (Haar Cascades work on grayscale).
2. Detects faces with `face_cascade.detectMultiScale()`.
3. For every face, draws a box labelled **Face**.
4. Cuts out the face region (ROI) and searches for smiles **only inside it** (lower half of the face).
5. Converts smile coordinates back to the full-frame coordinates (`x + sx`, `roi_top + sy`) and draws a box labelled **Smile**.

Parameters used:

| Parameter | Meaning |
|---|---|
| `scaleFactor` | How much the image is shrunk at each scan step (bigger = faster, less accurate) |
| `minNeighbors` | How many neighbouring detections are needed to accept one (bigger = fewer false positives) |
| `minSize` | Smallest object size to detect |

The smile detector uses strict values (`scaleFactor=1.7`, `minNeighbors=20`) because it often gives false positives. These values reduce false detections but do **not** guarantee perfect results.

In [16]:
def detect_faces_and_smiles(frame, face_cascade, smile_cascade):
    """Detect faces, then smiles inside each face.

    Returns (annotated_frame, number_of_faces, number_of_smiles).
    `frame` must be a BGR image (the OpenCV default).
    """
    output = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Step 1: find faces in the whole image
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.3, minNeighbors=5, minSize=(60, 60)
    )

    total_smiles = 0
    for (x, y, w, h) in faces:
        # Draw the face box and label
        cv2.rectangle(output, (x, y), (x + w, y + h), (255, 0, 0), 2)
        cv2.putText(output, "Face", (x, y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        # Step 2: search for smiles ONLY inside the lower half of the face
        # (smiles are never in the forehead/eyes; this reduces false positives)
        roi_top = y + h // 2
        face_gray = gray[roi_top:y + h, x:x + w]

        smiles = smile_cascade.detectMultiScale(
            face_gray, scaleFactor=1.7, minNeighbors=20, minSize=(25, 25)
        )

        for (sx, sy, sw, sh) in smiles:
            # Convert ROI coordinates back to full-frame coordinates
            fx, fy = x + sx, roi_top + sy
            cv2.rectangle(output, (fx, fy), (fx + sw, fy + sh), (0, 255, 0), 2)
            cv2.putText(output, "Smile", (fx, fy - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            total_smiles += 1

    return output, len(faces), total_smiles


def detect_smile(frame):
    """Simple wrapper: takes a BGR frame and returns only the processed frame."""
    result, _, _ = detect_faces_and_smiles(frame, face_cascade, smile_cascade)
    return result

## 7. Image-Based Smile Detection

Place an image named `sample.jpg` in the **same folder as this notebook**, or change `IMAGE_PATH` to any image you want to test.

In [17]:
IMAGE_PATH = "sample.jpg"   # <- put your image here (same folder as the notebook)

if not os.path.exists(IMAGE_PATH):
    print(f"Image not found: {IMAGE_PATH}. Place an image next to this notebook or change IMAGE_PATH.")
else:
    image = cv2.imread(IMAGE_PATH)
    if image is None:
        print("The file could not be read as an image.")
    else:
        result, n_faces, n_smiles = detect_faces_and_smiles(image, face_cascade, smile_cascade)

        # Matplotlib expects RGB, OpenCV uses BGR -> convert before showing
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title(f"Faces: {n_faces} | Smiles: {n_smiles}")
        plt.show()

Image not found: sample.jpg. Place an image next to this notebook or change IMAGE_PATH.


## 8. Webcam-Based Smile Detection

This cell opens your webcam with `cv2.VideoCapture(0)` and processes frames continuously:

```
Capture frame -> Grayscale -> Detect faces -> Detect smile in each face -> Draw boxes -> Display
```

**Press `q` to quit.**

Notes:
* This is intended for a **local** Python / Jupyter / VS Code environment.
* In Jupyter, the `cv2.imshow` window may open behind the browser; if it does not respond, close it with `q`, then restart the kernel.
* Google Colab cannot access your local webcam through `cv2.VideoCapture`, so use the Streamlit app instead.

In [18]:
def run_webcam():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Could not open the webcam.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to read a frame from the webcam.")
            break

        cv2.imshow("Smile Detection - press q to quit", detect_smile(frame))

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


run_webcam()

## 9. Testing / Examples

Test the pipeline on several images. Add your own file names to the list (for example one smiling photo, one neutral face, one without a face).

In [19]:
TEST_IMAGES = ["sample.jpg"]   # add more file names here

for path in TEST_IMAGES:
    img = cv2.imread(path) if os.path.exists(path) else None
    if img is None:
        print(f"{path}: not found or unreadable, skipped.")
        continue
    _, n_faces, n_smiles = detect_faces_and_smiles(img, face_cascade, smile_cascade)
    print(f"{path}: faces = {n_faces}, smiles = {n_smiles}")

sample.jpg: not found or unreadable, skipped.


## 10. Explanation of Results

* **Blue box + "Face"**: a face found by the face cascade.
* **Green box + "Smile"**: a smile found inside the lower half of a face.
* The count of faces and smiles is returned by the function and shown in the title/metrics.

Haar Cascade is a classical technique and can be affected by:

* Lighting
* Face angle
* Image quality / resolution
* Occlusion (glasses, masks, hands)
* Facial expression
* Camera quality

Because of this, you may sometimes see missed smiles or false detections. Tuning `minNeighbors` and `scaleFactor` trades one problem for the other.

## 11. Connecting to Streamlit

`app.py` contains the **same** `load_cascades()` and `detect_faces_and_smiles()` code as this notebook. It only adds a web interface around it:

| Notebook | Streamlit (`app.py`) |
|---|---|
| `cv2.imread()` | `st.file_uploader()` + PIL |
| `cv2.VideoCapture(0)` loop | `st.camera_input()` (single photo capture, not continuous video) |
| Matplotlib display | `st.image()` |

The app does not need the notebook to run. Start it from the project folder with:

```bash
streamlit run app.py
```

## 12. Conclusion

We built a smile detection system with only OpenCV's pretrained Haar Cascade classifiers: faces are detected first, then smiles are searched inside each face. The same logic works for images, a local webcam and a Streamlit web app, with no model training. The approach is simple and fast, but less accurate than modern deep-learning methods, which could be a future improvement.